# 01 Iron anode (IA-FE-001)

**Requirement:** `IA-FE-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


Discharge of metallic iron in alkaline electrolyte:

1. \(\mathrm{Fe + 2\,OH^- \leftrightarrow Fe(OH)_2 + 2\,e^-}\) with theoretical capacity **960 mAh/g Fe** (EP4602674A1 [0042]-[0043] Eq. 1).
2. \(3\,\mathrm{Fe(OH)_2 + 2\,OH^- \leftrightarrow Fe_3O_4 + 4\,H_2O + 2\,e^-}\) with **320 mAh/g Fe** (Eq. 2).

Faraday: \(r = I/(nF)\), \(n=2\). Butler-Volmer current from `ironair.chemistry.kinetics`.
Porosity absorbs the solid volume increase (IA-SYS-020). DRI beds, porous particles, and
channeled ribs (EP Claim 1; loading 1-7 g/cm², VF 0.5-0.9) share this RHS.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.iron_anode import IronAnode
from plant_sim.components.base import integrate_component
fe = IronAnode(P)
I = 40 * P.I_discharge_100h_A
t, y = integrate_component(lambda t, y: fe.rhs(t, y, {"I_fe_A": I, "T_amb_K": P.T_ep_sim_K, "a_oh": 6.0, "a_h2o": 0.72}, {}), fe.y0(), (0, 4000), n_eval=200)
timeseries(t/3600, {"n_Fe": y[0], "n_FeOH2": y[1], "n_Fe3O4": y[2]}, xlabel="t (h)", ylabel="mol", title="Iron phase inventories", callout="EP [0042]-[0043]", stem="nb01_phases")
cap = [fe.outputs(float(ti), y[:,i], {"I_fe_A": I, "a_oh": 6.0, "a_h2o": 0.72}, {})["capacity_extracted_mAh_g"] for i,ti in enumerate(t)]
print("extracted mAh/g (accelerated)", cap[-1], "theory 960 =", CAPACITY_FE_OH2_MAH_G)


## Verification against patents / SSS002

In [ ]:
assert abs(CAPACITY_FE_OH2_MAH_G - 960) < 1.0
assert abs(CAPACITY_MAGNETITE_MAH_G - 320) < 1.0
print("PASS Faraday capacities")
print("loading", P.loading_g_cm2, "g/cm2 in [1,7]; VF", P.vf_electrolyte_charged, "in [0.5,0.9]")
